# Example Usage of the Refactored MEG Analysis Pipeline

This notebook demonstrates how to use the refactored analysis pipeline for MEG sensor space analysis.

**Examples included:**
1. Single subject analysis
2. Batch processing multiple subjects
3. Group-level statistics
4. Visual inspection
5. Custom analysis with core functions

**Author:** 2026  
**Note:** Make sure `config.yaml` is properly configured and data files are available before running.

## 1. Import Required Libraries

Import all necessary modules for MEG analysis.

In [1]:
import os
import yaml
import numpy as np
import mne

# Import custom modules
from sensor_space_individual_analysis import run_individual_analysis
from sensor_space_group_statistics import run_group_statistics
from visual_inspection import VisualInspector
from STWM_functions_core import (
    load_preprocessed_data,
    create_epochs,
    compute_time_frequency,
    apply_fooof_multi_channel
)

/Users/administrator/__PC_HSE/Results/Nikita Otstavnov/SpatialTemporalWorkingMemoryMEG-main/visual_inspection.py:18: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof.plts.spectra import plot_spectrum, plot_spectra


## 2. Example 1: Single Subject Analysis

Process a single subject through the complete analysis pipeline.

In [2]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Modify for specific subject
# config['subject']['name'] = 'S1'
# config['subject']['file_name'] = '1_test_1_tsss_mc_trans.fif'

# Run analysis
print("Running individual analysis for subject S1...")
run_individual_analysis(config)
print("\n✓ Analysis complete!")

Running individual analysis for subject S1...

Starting Sensor Space Analysis for Subject: S1

Step 1: Loading preprocessed data...
  ✓ Loaded: sub-001_main2_cleaned2-raw.fif
  ✓ Duration: 2988.00 seconds
  ✓ Channels: 326 total

Step 2: Extracting events...
  ✓ Found 2211 events
  ✓ Condition 'SiVe' (ID=26): 60 events
  ✓ Condition 'CoVe' (ID=46): 61 events

Step 3: Creating epochs...
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
  ✓ Condition 'SiVe': 58 epochs
     Saved: S1_SiVe_epochs-epo.fif
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
  ✓ Condition 'CoVe': 55 epochs
     Saved: S1_CoVe_epochs-epo.fif
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
  ✓ Combined epochs: 113 epochs

Step 4: Computing time-frequency representations...
  ✓ Frequency range: 4.00 - 80.00 Hz
  ✓ Number of frequencies: 30
  ✓ Computing with 5 cycles...
Overwriting existing file.
Overwriting existing fil

/Users/administrator/__PC_HSE/Results/Nikita Otstavnov/SpatialTemporalWorkingMemoryMEG-main/meg_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overwriting existing file.
Overwriting existing file.
  ✓ Condition 'SiVe' CSD computed
Overwriting existing file.
Overwriting existing file.
  ✓ Condition 'CoVe' CSD computed

Analysis Complete for Subject: S1

Output files saved to: /Users/administrator/__PC_HSE/Results/Yulia K/Output

Generated files:
  - Epochs: *_epochs-epo.fif
  - Time-Frequency: *_power_*-tfr.h5, *_itc_*-tfr.h5
  - FOOOF: *_ped_crop.npy, *_aper_crop.npy
  - CSD: *_csd.h5

✓ Analysis complete!


## 3. Example 2: Batch Processing Multiple Subjects

Process multiple subjects in a loop.

In [ ]:
# Load base configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Define subjects
subjects = [
    {'name': 'S1', 'file': '1_test_1_tsss_mc_trans.fif'},
    {'name': 'S2', 'file': '2_test_1_tsss_mc_trans.fif'},
    {'name': 'S3', 'file': '3_test_1_tsss_mc_trans.fif'},
]

# Process each subject
for subj in subjects:
    print(f"\nProcessing {subj['name']}...")
    config['subject']['name'] = subj['name']
    config['subject']['file_name'] = subj['file']
    
    try:
        run_individual_analysis(config)
        print(f"✓ {subj['name']} completed")
    except Exception as e:
        print(f"✗ Error processing {subj['name']}: {e}")

## 4. Example 3: Group-Level Statistics

Run cluster-based permutation testing across all subjects.

In [ ]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set number of subjects
config['statistics']['num_subjects'] = 3

# Run group analysis
print("Running group statistics...")
T_obs, T_obs_plot, clusters, cluster_p_values = run_group_statistics(config)

# Display results
print("\n" + "="*60)
print("Results Summary")
print("="*60)
print(f"T-values shape: {T_obs.shape}")
print(f"Number of clusters: {len(clusters)}")
print(f"Significant clusters (p < 0.05): {np.sum(cluster_p_values < 0.05)}")

if np.any(cluster_p_values < 0.05):
    print("\nSignificant clusters:")
    for i, p in enumerate(cluster_p_values):
        if p < 0.05:
            print(f"  Cluster {i}: p = {p:.4f}")

## 5. Example 4: Visual Inspection

Load and visually inspect processed data for quality control.

In [ ]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Create inspector
inspector = VisualInspector(config)

# Load and inspect data
print("Loading epochs...")
epochs_S = inspector.load_and_inspect_epochs('S')
epochs_T = inspector.load_and_inspect_epochs('T')

print("\nLoading time-frequency data...")
power_S = inspector.load_and_inspect_power('S')
power_T = inspector.load_and_inspect_power('T')

print("\nCreating comparison report...")
inspector.create_comparison_report()

## 6. Example 5: Custom Analysis with Core Functions

Build a custom analysis pipeline using low-level core functions.

In [ ]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Load data
print("Loading data...")
folder = config['paths']['data_folder']
file_name = config['subject']['file_name']
file_path = os.path.join(folder, file_name)

raw_data = load_preprocessed_data(file_path, meg_type='grad')
print(f"✓ Loaded {len(raw_data.ch_names)} channels")

In [ ]:
# Extract events
print("Extracting events...")
events = mne.find_events(raw_data, stim_channel='STI101')
print(f"✓ Found {len(events)} events")

In [ ]:
# Create epochs
print("Creating epochs...")
epochs = create_epochs(
    raw_data, events, event_id=155,
    tmin=-8, tmax=8, picks='grad'
)
print(f"✓ Created {len(epochs)} epochs")

In [ ]:
# Compute time-frequency
print("Computing time-frequency...")
freqs = np.logspace(np.log10(4), np.log10(80), num=30)
power, itc = compute_time_frequency(
    epochs, freqs, n_cycles=5, decim=20, n_jobs=4
)
print(f"✓ TFR computed: {power.data.shape}")

In [ ]:
# Apply FOOOF
print("Applying FOOOF decomposition...")
spectrum = np.mean(power.data, axis=2)  # Average over time
periodic, aperiodic = apply_fooof_multi_channel(spectrum, freqs)
print(f"✓ FOOOF completed: periodic shape = {periodic.shape}")

print("\n✓ Custom analysis complete!")

## Notes

- Make sure to configure `config.yaml` before running any examples
- Examples can be run independently - just select the cell and run it
- For batch processing, ensure all data files exist in the specified folder
- Visual inspection will open interactive plots
- Group statistics requires all individual subjects to be processed first